# Perturbation & Statistical Validation

Covers:

1. Within-interview mismatch at `k = 1, 10, 50, 100`.
2. 100 cross-participant random-pairing permutations.
3. Participant-level mixed-effects analysis.
4. Holm correction across the four within-interview perturbation magnitudes.
5. Final Figure: SADI with 95% CI bars, plus Euclidean/cosine normalized change relative to matched.

In [ ]:
from pathlib import Path
import sys
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from scipy.stats import ttest_rel, wilcoxon
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42 # test different seeds
MODEL_PATH = PROJECT_ROOT / "models" / "v6_seed_best.pt"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "quantum" / "test.pkl"

RESULTS_DIR = PROJECT_ROOT / "notebooks" / "stats_q4" / "perturbation"
FIGURE_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("V6 checkpoint:", MODEL_PATH)
print("Test data:", TEST_PATH)
print("Perturbation results:", RESULTS_DIR)

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

from src.proposed.dataset import QuantumSequenceDataset, collate_fn
from src.proposed.v6.model_v6 import ProposedModelV6

print("✓ V6 imported.")

Device: cuda
✓ V6 imported.


In [ ]:
with open(TEST_PATH, "rb") as f:
    test_samples = pickle.load(f)

required = {
    "audio_hidden_states",
    "text_hidden_states",
    "participant_id",
    "label",
}

missing = required - set(test_samples[0])
if missing:
    raise KeyError(f"Missing required sequence fields: {sorted(missing)}")

print(f"Test records: {len(test_samples):,}")
print(f"Test participants: {len(set(str(x['participant_id']) for x in test_samples)):,}")

checkpoint = torch.load(MODEL_PATH, map_location=device)

if "model_state_dict" not in checkpoint:
    raise KeyError("Expected current train_v6.py checkpoint with 'model_state_dict'.")

model = ProposedModelV6(rank=16).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("✓ Proposed checkpoint loaded.")
print("Checkpoint seed:", checkpoint.get("seed", "unknown"))
print("Best epoch:", checkpoint.get("epoch", "unknown"))

In [6]:
def extract_v6_metrics(model, loader):
    model.eval()
    records = []

    with torch.no_grad():
        for batch in loader:
            audio = batch["audio"].to(device)
            text = batch["text"].to(device)
            audio_mask = batch["audio_mask"].to(device)
            text_mask = batch["text_mask"].to(device)

            out = model(audio, text, audio_mask, text_mask)

            labels = batch["labels"].cpu().numpy()
            pids = batch["participant_ids"]
            uids = batch["utterance_ids"]
            phq = batch["phq8_totals"].cpu().numpy()

            keys = [
                "sadi", "fidelity", "trace_distance",
                "audio_entropy", "text_entropy",
                "audio_purity", "text_purity",
                "joint_entropy_pre", "joint_entropy_post", "qmi"
            ]

            vals = {k: out[k].cpu().numpy() for k in keys}

            rho_t = out["rho_text_post"].cpu().numpy()
            rho_a = out["rho_audio_post"].cpu().numpy()

            for i in range(len(labels)):
                records.append({
                    "participant_id": str(pids[i]),
                    "utterance_id": str(uids[i]),
                    "label": int(labels[i]),
                    "phq8_total": float(phq[i]),
                    "discordance": float(vals["sadi"][i]),
                    "fidelity": float(vals["fidelity"][i]),
                    "trace_distance": float(vals["trace_distance"][i]),
                    "audio_entropy": float(vals["audio_entropy"][i]),
                    "text_entropy": float(vals["text_entropy"][i]),
                    "audio_purity": float(vals["audio_purity"][i]),
                    "text_purity": float(vals["text_purity"][i]),
                    "joint_entropy_pre": float(vals["joint_entropy_pre"][i]),
                    "joint_entropy_post": float(vals["joint_entropy_post"][i]),
                    "mutual_information": float(vals["qmi"][i]),
                    "rho_text_post": rho_t[i],
                    "rho_audio_post": rho_a[i],
                })

    return pd.DataFrame(records)


def summarise_participant_level(df):
    metrics = [
        "discordance", "fidelity", "trace_distance",
        "audio_entropy", "text_entropy",
        "audio_purity", "text_purity",
        "joint_entropy_pre", "joint_entropy_post",
        "mutual_information"
    ]

    agg = {"label": "first", "phq8_total": "first"}
    agg.update({m: "mean" for m in metrics})

    return (
        df.groupby("participant_id", as_index=False)
          .agg(agg)
          .sort_values("participant_id")
          .reset_index(drop=True)
    )


def vectorise_density_states(df):
    text = np.stack([np.asarray(x).reshape(-1) for x in df["rho_text_post"]])
    audio = np.stack([np.asarray(x).reshape(-1) for x in df["rho_audio_post"]])
    return text.astype(float), audio.astype(float)


def calculate_classical_distances(df):
    text, audio = vectorise_density_states(df)

    euclidean = np.linalg.norm(text - audio, axis=1)

    cosine = 1.0 - (
        np.sum(text * audio, axis=1)
        /
        (
            np.linalg.norm(text, axis=1)
            * np.linalg.norm(audio, axis=1)
            + 1e-12
        )
    )

    out = df.copy()
    out["euclidean_distance"] = euclidean
    out["cosine_distance"] = cosine
    return out


class InMemoryQuantumSequenceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx]
        return {
            "audio": torch.tensor(np.asarray(x["audio_hidden_states"], dtype=np.float32)),
            "text": torch.tensor(np.asarray(x["text_hidden_states"], dtype=np.float32)),
            "label": torch.tensor(int(x["label"]), dtype=torch.long),
            "participant_id": str(x["participant_id"]),
            "utterance_id": str(x.get("utterance_id", f"perturb_{idx}")),
            "phq8_total": torch.tensor(float(x.get("phq8_total", np.nan)), dtype=torch.float32),
        }


def collate_mismatch_fn(batch):
    B = len(batch)
    audio_lengths = [x["audio"].shape[0] for x in batch]
    text_lengths = [x["text"].shape[0] for x in batch]

    A = torch.zeros(B, max(audio_lengths), 768, dtype=torch.float32)
    T = torch.zeros(B, max(text_lengths), 768, dtype=torch.float32)
    am = torch.zeros(B, max(audio_lengths), dtype=torch.bool)
    tm = torch.zeros(B, max(text_lengths), dtype=torch.bool)

    labels, pids, uids, phq = [], [], [], []

    for i, x in enumerate(batch):
        a, t = x["audio"], x["text"]
        A[i, :a.shape[0]] = a
        T[i, :t.shape[0]] = t
        am[i, :a.shape[0]] = True
        tm[i, :t.shape[0]] = True
        labels.append(x["label"])
        pids.append(x["participant_id"])
        uids.append(x["utterance_id"])
        phq.append(x["phq8_total"])

    return {
        "audio": A,
        "text": T,
        "audio_mask": am,
        "text_mask": tm,
        "lengths_audio": torch.tensor(audio_lengths, dtype=torch.long),
        "lengths_text": torch.tensor(text_lengths, dtype=torch.long),
        "labels": torch.stack(labels),
        "participant_ids": pids,
        "utterance_ids": uids,
        "phq8_totals": torch.stack(phq),
    }


def evaluate_samples(samples):
    ds = InMemoryQuantumSequenceDataset(samples)

    loader = DataLoader(
        ds,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_mismatch_fn,
        pin_memory=(device.type == "cuda"),
    )

    return extract_v6_metrics(model, loader)

Use the original E-DAIC test pairing as the fixed reference for every perturbation.

In [ ]:
correct_cache = PROJECT_ROOT / "notebooks" / "stats_q4" / "correct_pair_metrics.pkl"

if correct_cache.exists():
    correct_full = pd.read_pickle(correct_cache)
    print("Loaded cached matched segment metrics:", correct_cache)
else:
    test_dataset = QuantumSequenceDataset(TEST_PATH)
    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=(device.type == "cuda"),
    )
    correct_full = extract_v6_metrics(model, test_loader)
    correct_full.to_pickle(correct_cache)

correct_full = calculate_classical_distances(correct_full)
correct_participant = summarise_participant_level(correct_full)

correct_dist = (
    correct_full[
        ["participant_id", "euclidean_distance", "cosine_distance"]
    ]
    .groupby("participant_id", as_index=False)
    .mean()
)

correct_participant_dist = correct_participant.merge(
    correct_dist,
    on="participant_id",
    how="left"
)

assert len(correct_participant_dist) == 56

print("Matched records:", len(correct_full))
print("Matched participants:", len(correct_participant_dist))
display(correct_participant_dist.head())

## Within-interview mismatch: `k = 1, 10, 50, 100`

Speech is paired with a segment `k` positions away **within the same participant interview**. A circular shift is used to retain the interview-local structure. Self-pairs are excluded if a particular participant's sequence length makes the requested offset map back to the same segment.

In [8]:
WITHIN_SHIFTS = [1, 10, 50, 100]


def within_interview_shift(samples, shift):
    by_pid = {}

    for x in samples:
        by_pid.setdefault(str(x["participant_id"]), []).append(x)

    out = []
    excluded = 0

    for pid, items in by_pid.items():
        n = len(items)

        for i, x in enumerate(items):
            j = (i + shift) % n

            if j == i:
                excluded += 1
                continue

            source = items[j]
            y = dict(x)

            y["audio_hidden_states"] = np.asarray(
                source["audio_hidden_states"]
            ).copy()

            y["_audio_source_participant"] = pid
            y["_audio_source_utterance"] = str(
                source.get("utterance_id", j)
            )
            y["_within_shift"] = int(shift)

            out.append(y)

    return out, excluded


within_samples = {}
within_excluded = {}

for k in WITHIN_SHIFTS:
    samples_k, excluded_k = within_interview_shift(test_samples, k)

    within_samples[k] = samples_k
    within_excluded[k] = excluded_k

    same_participant = sum(
        str(x["participant_id"]) == str(x["_audio_source_participant"])
        for x in samples_k
    )

    print(
        f"k={k:3d} | records={len(samples_k):,} | "
        f"excluded={excluded_k:,} | "
        f"same-participant pairs={same_participant:,}"
    )

    assert same_participant == len(samples_k)

k=  1 | records=5,081 | excluded=0 | same-participant pairs=5,081
k= 10 | records=5,081 | excluded=0 | same-participant pairs=5,081
k= 50 | records=4,981 | excluded=100 | same-participant pairs=4,981
k=100 | records=4,981 | excluded=100 | same-participant pairs=4,981


In [ ]:
within_segment = {}
within_participant = {}

for k in WITHIN_SHIFTS:
    cache = RESULTS_DIR / f"within_shift_{k}_metrics.pkl"

    if cache.exists():
        df_k = pd.read_pickle(cache)
        print(f"Loaded cached k={k} results.")
    else:
        print(f"Evaluating within-interview shift k={k}...")
        df_k = evaluate_samples(within_samples[k])
        df_k.to_pickle(cache)

    df_k = calculate_classical_distances(df_k)
    within_segment[k] = df_k

    p_k = summarise_participant_level(df_k)

    dist_k = (
        df_k[
            ["participant_id", "euclidean_distance", "cosine_distance"]
        ]
        .groupby("participant_id", as_index=False)
        .mean()
    )

    p_k = p_k.merge(dist_k, on="participant_id", how="left")
    within_participant[k] = p_k

    print(
        f"  records={len(df_k):,}, "
        f"participants={len(p_k):,}"
    )

within_rows = []

for k in WITHIN_SHIFTS:
    for _, row in within_participant[k].iterrows():
        within_rows.append({
            "shift": k,
            "participant_id": row["participant_id"],
            "discordance": row["discordance"],
            "fidelity": row["fidelity"],
            "euclidean_distance": row["euclidean_distance"],
            "cosine_distance": row["cosine_distance"],
        })

within_participant_long = pd.DataFrame(within_rows)

within_participant_long.to_csv(
    RESULTS_DIR / "within_interview_participant_metrics.csv",
    index=False
)

pd.DataFrame([
    {
        "shift": k,
        "records": len(within_segment[k]),
        "excluded_self_pairs": within_excluded[k],
        "participants": len(within_participant[k]),
    }
    for k in WITHIN_SHIFTS
]).to_csv(
    RESULTS_DIR / "within_interview_construction_summary.csv",
    index=False
)

display(within_participant_long.head())

## Cross-participant mismatch: 100 random permutations

Each permutation pairs every participant's text with speech drawn from a **different participant**. The participant mapping is fixed-point-free. Each completed permutation is cached so the notebook can resume after interruption.

In [10]:
N_CROSS_PERMUTATIONS = 100
CROSS_BASE_SEED = 202500


def cross_participant_mismatch(samples, seed):
    rng = np.random.default_rng(seed)

    pids = np.asarray([str(x["participant_id"]) for x in samples])
    unique = np.unique(pids)

    if len(unique) < 2:
        raise ValueError("Need at least two participants.")

    for _ in range(1000):
        perm = rng.permutation(unique)
        mapping = dict(zip(unique, perm))

        if all(mapping[p] != p for p in unique):
            break
    else:
        raise RuntimeError(
            "Could not construct a fixed-point-free participant pairing."
        )

    by_pid = {}
    for x in samples:
        by_pid.setdefault(str(x["participant_id"]), []).append(x)

    out = []

    for x in samples:
        original_pid = str(x["participant_id"])
        source_pid = str(mapping[original_pid])
        source_items = by_pid[source_pid]

        source_idx = int(rng.integers(len(source_items)))
        source = source_items[source_idx]

        y = dict(x)
        y["audio_hidden_states"] = np.asarray(
            source["audio_hidden_states"]
        ).copy()

        y["_audio_source_participant"] = source_pid
        y["_audio_source_utterance"] = str(
            source.get("utterance_id", source_idx)
        )
        y["_cross_permutation"] = int(seed)

        out.append(y)

    return out

In [ ]:
# ============================================================
# CROSS-PARTICIPANT PERTURBATION
# 100 RANDOM PERMUTATIONS
#
# Lightweight JSON checkpoint every 10 permutations.
#
# Retains PARTICIPANT-LEVEL results only.
# Segment-level V6 outputs are not persisted for every permutation because they are unnecessary for the later mixed-effects / 
# Holm analysis and consume substantial disk.

import json
import numpy as np
import pandas as pd
from pathlib import Path


N_CROSS_PERMUTATIONS = 100
CROSS_BASE_SEED = 2025

CHECKPOINT_EVERY = 10

RESULTS_DIR = Path(
    "data/results/perturbation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR = (
    RESULTS_DIR / "cross_participant_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def save_cross_checkpoint(
    cross_perm_participant,
    completed_through
):

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"cross_participant_checkpoint_{completed_through:03d}.json"
    )

    payload = {
        "experiment": "Cross-participant perturbation",
        "completed_through": int(completed_through),
        "n_permutations": int(N_CROSS_PERMUTATIONS),
        "base_seed": int(CROSS_BASE_SEED),
        "checkpoint_every": int(CHECKPOINT_EVERY),
        "results": {}
    }

    for perm_idx, p_df in sorted(
        cross_perm_participant.items()
    ):

        # Convert NaN/inf safely for JSON
        clean_df = p_df.copy()

        clean_df = clean_df.replace(
            [np.inf, -np.inf],
            np.nan
        )

        records = json.loads(
            clean_df.to_json(
                orient="records"
            )
        )

        payload["results"][str(perm_idx)] = records


    with open(
        checkpoint_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            allow_nan=False
        )


    print(
        f"  ✓ checkpoint saved: "
        f"{checkpoint_path.name}"
    )

    return checkpoint_path


# ------------------------------------------------------------
# Helper: load latest checkpoint
# ------------------------------------------------------------

def load_latest_cross_checkpoint():

    checkpoints = sorted(
        CHECKPOINT_DIR.glob(
            "cross_participant_checkpoint_*.json"
        )
    )

    if not checkpoints:
        return {}, 0


    latest = checkpoints[-1]

    print(
        f"Loading latest checkpoint: "
        f"{latest.name}"
    )


    with open(
        latest,
        "r",
        encoding="utf-8"
    ) as f:

        payload = json.load(f)


    cross_perm_participant = {}


    for perm_key, records in payload[
        "results"
    ].items():

        perm_idx = int(perm_key)

        p_df = pd.DataFrame(records)

        if "participant_id" in p_df.columns:

            p_df["participant_id"] = (
                p_df["participant_id"]
                .astype(str)
            )

        cross_perm_participant[
            perm_idx
        ] = p_df


    completed_through = int(
        payload["completed_through"]
    )


    print(
        f"  ✓ loaded "
        f"{len(cross_perm_participant)} "
        f"completed permutations"
    )

    print(
        f"  ✓ resume point: "
        f"permutation {completed_through + 1}"
    )


    return (
        cross_perm_participant,
        completed_through
    )


# ============================================================
# LOAD EXISTING CHECKPOINT

(
    cross_perm_participant,
    last_completed
) = load_latest_cross_checkpoint()


# ============================================================
# RUN REMAINING PERMUTATIONS

for perm_idx in range(
    last_completed + 1,
    N_CROSS_PERMUTATIONS + 1
):

    seed = (
        CROSS_BASE_SEED +
        perm_idx
    )


    print(
        "\n"
        + "=" * 72
    )

    print(
        f"Cross-participant permutation "
        f"{perm_idx}/{N_CROSS_PERMUTATIONS}"
    )

    print(
        f"Seed: {seed}"
    )

    print(
        "=" * 72
    )


    # --------------------------------------------------------
    # Construct cross-participant mismatch
    # --------------------------------------------------------

    samples = cross_participant_mismatch(
        test_samples,
        seed=seed
    )


    # --------------------------------------------------------
    # Verify that NO audio comes from the same participant
    # --------------------------------------------------------

    same = sum(
        str(x["participant_id"]) ==
        str(x["_audio_source_participant"])
        for x in samples
    )


    if same != 0:

        raise AssertionError(
            f"Permutation {perm_idx} contains "
            f"{same} same-participant pairs."
        )


    print(
        f"✓ {len(samples):,} records"
    )

    print(
        "✓ 0 same-participant pairs"
    )


    # --------------------------------------------------------
    # Evaluate V6
    # --------------------------------------------------------

    print(
        "Evaluating V6..."
    )

    df = evaluate_samples(
        samples
    )


    # --------------------------------------------------------
    # Classical distances using the same vectors
    # --------------------------------------------------------

    df = calculate_classical_distances(
        df
    )


    # --------------------------------------------------------
    # Participant-level aggregation
    # --------------------------------------------------------

    p_df = summarise_participant_level(
        df
    )


    # --------------------------------------------------------
    # Participant-level Euclidean / cosine
    # --------------------------------------------------------

    dist_df = (
        df[
            [
                "participant_id",
                "euclidean_distance",
                "cosine_distance"
            ]
        ]
        .groupby(
            "participant_id",
            as_index=False
        )
        .mean()
    )


    p_df = p_df.merge(
        dist_df,
        on="participant_id",
        how="left"
    )


    # --------------------------------------------------------
    # Add permutation identifier
    # --------------------------------------------------------

    p_df["permutation"] = (
        perm_idx
    )

    p_df["permutation_seed"] = (
        seed
    )


    # --------------------------------------------------------
    # Store ONLY participant-level result in memory
    # --------------------------------------------------------

    cross_perm_participant[
        perm_idx
    ] = p_df


    print(
        f"✓ participant-level result: "
        f"{len(p_df)} participants"
    )


    # --------------------------------------------------------
    # CHECKPOINT EVERY 10 PERMUTATIONS
    # --------------------------------------------------------

    if (
        perm_idx % CHECKPOINT_EVERY == 0
        or perm_idx == N_CROSS_PERMUTATIONS
    ):

        save_cross_checkpoint(
            cross_perm_participant,
            completed_through=perm_idx
        )


# ============================================================
# FINAL COMBINED PARTICIPANT-LEVEL DATAFRAME

cross_participant_all = pd.concat(
    cross_perm_participant.values(),
    ignore_index=True
)


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

cross_participant_all = (
    cross_participant_all
    .sort_values(
        [
            "permutation",
            "participant_id"
        ]
    )
    .reset_index(drop=True)
)

# Save one final CSV as well-

final_csv = (
    RESULTS_DIR /
    "cross_participant_100_permutations_participant_level.csv"
)


cross_participant_all.to_csv(
    final_csv,
    index=False
)


print(
    "\n"
    + "=" * 72
)

print(
    "✓ CROSS-PARTICIPANT PERTURBATION COMPLETE"
)

print(
    "=" * 72
)

print(
    f"Permutations: "
    f"{cross_participant_all['permutation'].nunique()}"
)

print(
    f"Participant-level rows: "
    f"{len(cross_participant_all):,}"
)

print(
    f"Participants per permutation: "
    f"{cross_participant_all.groupby('permutation').size().iloc[0]}"
)

print(
    f"Saved final participant-level data:"
)

print(
    final_csv
)

## Participant-level mixed-effects analysis

Perform the inferential comparison at the **participant level** with:

`metric ~ condition + (1 | participant)`

For the cross-participant condition, the 100 random permutations are first averaged within participant. This prevents the 100 permutations from inflating the nominal number of participants.

In [13]:
def fit_participant_mixed_effect(
    matched_df,
    perturbed_df,
    metric="discordance",
    condition_name="Mismatch"
):
    a = matched_df[["participant_id", metric]].copy()
    b = perturbed_df[["participant_id", metric]].copy()

    a["condition"] = 0
    b["condition"] = 1

    data = pd.concat([a, b], ignore_index=True)
    data["participant_id"] = data["participant_id"].astype(str)
    data = data.dropna(subset=[metric, "participant_id"])

    fit = mixedlm(
        f"{metric} ~ condition",
        data=data,
        groups=data["participant_id"],
        re_formula="1"
    ).fit(
        reml=False,
        method="lbfgs",
        maxiter=1000,
        disp=False
    )

    estimate = float(fit.params["condition"])
    ci = fit.conf_int().loc["condition"]

    return {
        "condition": condition_name,
        "effect_estimate": estimate,
        "95CI_lower": float(ci.iloc[0]),
        "95CI_upper": float(ci.iloc[1]),
        "p_value": float(fit.pvalues["condition"]),
        "n_participants": data["participant_id"].nunique(),
    }

In [ ]:
within_mixed_rows = []

RESULTS_DIR = PROJECT_ROOT / "notebooks" / "stats_q4" / "perturbation"

for k in WITHIN_SHIFTS:
    result = fit_participant_mixed_effect(
        correct_participant_dist,
        within_participant[k],
        metric="discordance",
        condition_name=f"Within-interview k={k}"
    )

    result["shift"] = k
    within_mixed_rows.append(result)

within_mixed_df = pd.DataFrame(within_mixed_rows)

display(within_mixed_df)

In [17]:
# ============================================================
# Holm correction across k=1, 10, 50, 100

reject, p_holm, _, _ = multipletests(
    within_mixed_df["p_value"].to_numpy(),
    alpha=0.05,
    method="holm"
)

holm_df = within_mixed_df.copy()
holm_df["p_holm"] = p_holm
holm_df["significant_after_holm"] = reject

holm_df.to_csv(
    RESULTS_DIR / "within_interview_holm_correction.csv",
    index=False
)

display(holm_df)

,condition,effect_estimate,95CI_lower,95CI_upper,p_value,n_participants,shift,p_holm,significant_after_holm
0,Within-interview k=1,0.000590,0.000227,0.000953,0.001442,56,1,0.005770,True
1,Within-interview k=10,0.000581,0.000115,0.001047,0.014528,56,10,0.034401,True
2,Within-interview k=50,0.000546,0.000104,0.000989,0.015563,56,50,0.034401,True
3,Within-interview k=100,0.000559,0.000126,0.000992,0.011467,56,100,0.034401,True


In [ ]:
# AVERAGE 100 CROSS-PARTICIPANT PERMUTATIONS
# WITHIN PARTICIPANT
# ============================================================

# Combine the participant-level results from all permutations
cross_perm_long = pd.concat(
    cross_perm_participant.values(),
    ignore_index=True
)

print(
    f"Loaded {cross_perm_long['permutation'].nunique()} permutations"
)

print(
    f"Total participant-level rows: "
    f"{len(cross_perm_long):,}"
)


# ------------------------------------------------------------
# Average across the 100 permutations for each participant
# ------------------------------------------------------------

cross_participant_summary = (
    cross_perm_long
    .groupby(
        "participant_id",
        as_index=False
    )[
        [
            "discordance",
            "fidelity",
            "euclidean_distance",
            "cosine_distance",
        ]
    ]
    .mean()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

summary_path = (
    RESULTS_DIR /
    "cross_participant_100_perm_participant_mean.csv"
)

cross_participant_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    f"\n✓ Saved participant-level permutation mean:"
)

print(
    summary_path.resolve()
)

print(
    f"✓ Participants: "
    f"{len(cross_participant_summary)}"
)

display(
    cross_participant_summary.head()
)

In [ ]:
# ============================================================
# CROSS-PARTICIPANT MIXED-EFFECTS DATA
# Keep all 100 permutations as repeated observations.

cross_mixed_long = pd.concat(
    cross_perm_participant.values(),
    ignore_index=True
)

cross_mixed_long["participant_id"] = (
    cross_mixed_long["participant_id"].astype(str)
)

cross_mixed_long["condition"] = (
    "Cross-participant"
)

print(
    "Rows:",
    len(cross_mixed_long)
)

print(
    "Participants:",
    cross_mixed_long["participant_id"].nunique()
)

print(
    "Permutations:",
    cross_mixed_long["permutation"].nunique()
)

display(
    cross_mixed_long[
        [
            "participant_id",
            "permutation",
            "discordance"
        ]
    ].head()
)

## Cross-participant permutation uncertainty

In addition to the mixed-effects result, retain the distribution of the participant-level SADI effect across all 100 random pairings.

In [ ]:
matched_by_pid = correct_participant_dist.set_index("participant_id")

cross_perm_effect_rows = []

for perm, df in cross_perm_participant.items():
    tmp = df.set_index("participant_id")

    common = matched_by_pid.index.intersection(tmp.index)

    d = (
        tmp.loc[common, "discordance"]
        -
        matched_by_pid.loc[common, "discordance"]
    )

    cross_perm_effect_rows.append({
        "permutation": perm,
        "mean_sadi_difference": float(d.mean()),
        "sd_sadi_difference": float(d.std(ddof=1)),
    })

cross_perm_effect_df = pd.DataFrame(
    cross_perm_effect_rows
)

cross_perm_effect_df.to_csv(
    RESULTS_DIR / "cross_participant_permutation_effects.csv",
    index=False
)

display(cross_perm_effect_df.describe())